# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [27]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [28]:
import os
import numpy as np
import dask.dataframe as dd
from glob import glob

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [29]:
# Write your code below.
# Load env var set via dotenv 
PRICE_DATA = os.getenv("PRICE_DATA")


In [30]:
if not PRICE_DATA:
    raise EnvironmentError("PRICE_DATA is not set. Did you run %load_ext dotenv and %dotenv?")
if not os.path.isdir(PRICE_DATA):
    raise FileNotFoundError(f"PRICE_DATA path does not exist or is not a directory: {PRICE_DATA}")


In [31]:
# Find all parquet file paths under PRICE_DATA, recursively
parquet_files = glob(os.path.join(PRICE_DATA, "**", "*.parquet"), recursive=True)

# Quick peek so you can see it worked
len(parquet_files), (parquet_files[:5] if parquet_files else parquet_files)

(3071,
 ['../../05_src/data/prices\\AACG\\AACG_2019\\part.0.parquet',
  '../../05_src/data/prices\\AACG\\AACG_2019\\part.1.parquet',
  '../../05_src/data/prices\\AACG\\AACG_2020\\part.0.parquet',
  '../../05_src/data/prices\\AACG\\AACG_2020\\part.1.parquet',
  '../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet'])

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [32]:
# Write your code below.

# Build price_dd from the discovered parquet files
price_dd = dd.read_parquet(parquet_files).set_index("ticker")
price_dd["Date"] = dd.to_datetime(price_dd["Date"])


In [33]:
price_dd

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=90,,,,,,,,,
AACG,datetime64[ns],float64,float64,float64,float64,float64,int64,string,int32
ACN,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


In [34]:
price_dd.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year'],
      dtype='object')

In [35]:
# Change the name"Adj Close" to "Adj_Close"
price_dd = price_dd.rename(columns=lambda c: c.replace(" ", "_"))
# Bring 'ticker' into a column so we can (re)index on it
ddf = price_dd.reset_index()  # 'ticker' becomes a column

In [36]:
ddf

,ticker,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year
npartitions=90,,,,,,,,,,
,string,datetime64[ns],float64,float64,float64,float64,float64,int64,string,int32
,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...


In [37]:
# Collect sorted unique tickers
tickers = ddf["ticker"].unique().compute().tolist()
tickers = sorted(tickers)

In [38]:
tickers

['AACG',
 'ACN',
 'ADP',
 'AKBA',
 'ALDX',
 'ALL',
 'AMAL',
 'AMH',
 'ANIX',
 'AQMS',
 'ARE',
 'BCBP',
 'BGS',
 'BGSF',
 'BLPH',
 'BOMN',
 'BPMX',
 'BPYPN',
 'BRQS',
 'BWEN',
 'BWG',
 'BXS',
 'CBB',
 'CGEN',
 'CLSD',
 'CMCTP',
 'CRMT',
 'CSSE',
 'DIVA',
 'EARN',
 'EOLS',
 'ERH',
 'ESGR',
 'ETJ',
 'EXLS',
 'FAMI',
 'FDIS',
 'FIXX',
 'GAZ',
 'GLADD',
 'GLUU',
 'GLW',
 'GURE',
 'HMHC',
 'IAU',
 'INSU',
 'IPWR',
 'ITCB',
 'KALU',
 'KEY',
 'KWR',
 'LEVL',
 'LH',
 'MMAC',
 'MNK',
 'MOH',
 'MOS',
 'MUNI',
 'NGD',
 'NPK',
 'PFG',
 'QRHC',
 'REG',
 'REI',
 'RFDI',
 'RGEN',
 'RIV',
 'RTTR',
 'SLRX',
 'SMG',
 'SMMT',
 'SPXC',
 'SRE',
 'SYBX',
 'SYNH',
 'TEF',
 'THQ',
 'TMP',
 'TNC',
 'TRIL',
 'TRTX',
 'TSN',
 'VIAC',
 'WEX',
 'WORK',
 'WST',
 'XBI',
 'XFLT',
 'XOM',
 'ZIXI']

In [39]:
divisions = tickers + ['\uffff']  # a unicode sentinel greater than typical ticker strings

In [40]:
# Set index with explicit divisions (creates ~one partition per ticker)
ddf = ddf.set_index("ticker", shuffle="tasks", divisions=divisions)

In [41]:
# Define a per-partition (i.e., per-ticker) function to build all features
def _build_features(pdf):
    # Ensure chronological order inside the ticker;
    pdf = pdf.sort_values("Date")
    # 1-day lags: yesterday's values aligned to today's row.
    pdf["Close_lag_1"] = pdf["Close"].shift(1)
    pdf["Adj_Close_lag_1"] = pdf["Adj_Close"].shift(1)
    # Daily simple return based on Close:
    # return_t = Close_t / Close_{t-1} - 1
    pdf["returns"]  = (pdf["Close"] / pdf["Close_lag_1"]) - 1
    # Intraday high-low range 
    pdf["hi_lo_range"] = pdf["High"] - pdf["Low"]
    # 10-day moving average of returns.
    pdf["ma10_returns"] = pdf["returns"].rolling(10).mean()

    # Return the partition (i.e., the single-ticker DataFrame) with new columns added.
    return pdf

In [42]:
# Build an explicit Dask `meta` (schema) so groupby/apply/map_partitions
# know the output column names and dtypes *without* executing the full task graph.
# This prevents "meta" inference errors and speeds up graph construction.

_meta = (
    ddf._meta.assign(                  # start from the CURRENT schema of `ddf`
        Close_lag_1=np.float64(),      # new float column: 1-day lag of Close
        Adj_Close_lag_1=np.float64(),  # new float column: 1-day lag of Adj_Close
        returns=np.float64(),          # new float column: (Close / Close_lag_1) - 1
        hi_lo_range=np.float64(),      # new float column: High - Low
        ma10_returns=np.float64(),     # new float column: 10-day rolling mean of returns
    )
)

In [43]:
# Build per-ticker features using groupby+apply.
# This computes lags/returns within each ticker after sorting by Date.
dd_feat = (
    price_dd
    .groupby("ticker", group_keys=False)
    .apply(
        # For each ticker's chunk (as a pandas DataFrame `x`):
        lambda x: x.sort_values("Date").assign(
            # 1-day lags
            Close_lag_1 = x["Close"].shift(1),
            Adj_Close_lag_1 = x["Adj_Close"].shift(1),
            returns = (x["Close"] / x["Close"].shift(1)) - 1,
            # Intraday range in price units
            hi_lo_range = x["High"] - x["Low"],
        ),
        meta=_meta
    )
)
# RESULT: `dd_feat` is a Dask DataFrame with new columns:
#   Close_lag_1, Adj_Close_lag_1, returns, hi_lo_range

In [44]:
# Apply per-ticker feature builder; result stays in Dask
dd_feat = ddf.map_partitions(_build_features, meta=_meta)

In [45]:
dd_feat.head()

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,ma10_returns
ticker,,,,,,,,,,,,,,
AACG,2019-03-11,1.08,1.10,1.08,1.08,1.08,32100.0,AACG.csv,2019,NaN,NaN,NaN,0.02,NaN
AACG,2019-03-12,1.08,1.09,1.05,1.05,1.05,20200.0,AACG.csv,2019,1.08,1.08,-0.027778,0.04,NaN
AACG,2019-03-13,1.06,1.08,1.04,1.07,1.07,23100.0,AACG.csv,2019,1.05,1.05,0.019048,0.04,NaN
AACG,2019-03-14,1.06,1.11,1.06,1.08,1.08,29900.0,AACG.csv,2019,1.07,1.07,0.009346,0.05,NaN
AACG,2019-03-15,1.06,1.08,1.04,1.04,1.04,30900.0,AACG.csv,2019,1.08,1.08,-0.037037,0.04,NaN


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [46]:
# Convert Dask → pandas (make ticker a column so groupby works)
feat_pd = dd_feat.reset_index().compute()

# Ensure per-ticker chronological order (good hygiene)
feat_pd = feat_pd.sort_values(["ticker", "Date"])

# Add 10-day moving average of returns per ticker
feat_pd["ma10_returns"] = (
    feat_pd.groupby("ticker")["returns"]
           .transform(lambda s: s.rolling(10).mean())
)

# (Optional) quick peek
feat_pd[["ticker", "Date", "returns", "ma10_returns"]].head(15)




,ticker,Date,returns,ma10_returns
0,AACG,2019-03-11,NaN,NaN
1,AACG,2019-03-12,-0.027778,NaN
2,AACG,2019-03-13,0.019048,NaN
3,AACG,2019-03-14,0.009346,NaN
4,AACG,2019-03-15,-0.037037,NaN
5,AACG,2019-03-18,0.028846,NaN
6,AACG,2019-03-19,-0.009346,NaN
7,AACG,2019-03-20,-0.047170,NaN
8,AACG,2019-03-21,0.019802,NaN
9,AACG,2019-03-22,0.000000,NaN


Please comment:

*+ Was it necessary to convert to pandas to calculate the moving average return?*

No. We could compute a 10-day moving average in Dask

+ Would it have been better to do it in Dask? Why?

Yes, Dask is preferable here because:
- It scales to bigger-than-RAM data (future-proof).
- It preserves a lazy, composable pipeline (scheduling, logging, retries).

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.